# Template for Agent Evaluation

_Assessing agents performance._

Build an agent with mock tools and evaluate the agent by assessing (1) its execution trajectory and (2) the sequence of messages and tool calls it produces, by using an built-in evaluator (that takes agent outputs and optionally reference outputs, and returns a score) over approaches such as _Trajectory Match_ (for deterministic comparison) and _LLM-as-judge_ (for qualitative assessment).


**Prerequisites:**

Follow the instruction below.

1. Open a new **Anaconda Prompt** terminal and activate environment `agentic_ai` using command `conda activate agentic_ai`.

2. Refer to the package installtion section in GitHub README.md and install LangChain `agentevals` in the same environment.

In [ ]:
# Imports packages

# Import class `Literal` from module `typing`
# Import `tool` from module `langchain.tools`
# Import class `ChatOllama` from module `langchain_ollama`
# Import function `create_agent` from module `langchain.agents`
# Import class `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` from module `langchain.message`s
# Iimport function `create_trajectory_match_evaluator` from module `agentevals.trajectory`
# Import function `create_trajectory_llm_as_judge`, and 
# constant `TRAJECTORY_ACCURACY_PROMPT` and `TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE` 
# from module `agentevals.trajectory.llm`

## Model

_Connecting an appropriate model to a chat client._

In [3]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_MODEL_AS_JUDGE = "granite4.1:3b"     # "qwen3.5:2b" found causing error in generating data in valid JSON
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [ ]:
# Initialize a chat client by calling constuctor of class ChatOllama with the following arguments.
# 1. model name against parameter `model`, 
# 2. 'False` against parameter `reasoning`, and
# 3. model endpoint to parameter `base_url`, and
# store the reference to the created instance in a variable called `model`.

# Code here
# ...


# Similarly, initialize a model client to be used as judge during evaluation 
# by calling constuctor of class ChatOllama with the following arguments.
# 1. the judge model name against parameter `model`, 
# 2. 'False` against parameter `reasoning`, and
# 3. model endpoint to parameter `base_url`, and
# store the reference to the created instance in a variable called `model_as_judge`.

# Code here
# ...

## Tools

_Tools the agent needs to complete its tasks._

For this experiment, all tools are dummy. But actual function implementation would be required for production use cases.

In [5]:
@tool
def get_weather(city: str):
    """
    Get weather information for a city.
    
    Parameters:
    - city (str): The name of the city to get weather information for.
    """
    
    return f"It's 75 degrees and sunny in {city}."


@tool
def get_events(city: str):
    """
    Get events happening in a city.
    
    Parameters:
    - city (str): The name of the city to get events for.
    """
    
    return f"Concert at the park in {city} tonight."


@tool
def get_detailed_forecast(city: str):
    """
    Get detailed weather forecast for a city.
    
    Parameters:
    - city (str): The name of the city to get detailed forecast for.
    """
    
    return f"Detailed forecast for {city}: sunny all week."

In [ ]:
# Create an agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing tool function `get_weather`, `get_events` and `get_detailed_forecast` against parameter `tools`, and
# store the reference of the created instance into a variable called `agent`.

# Code here
# ...

## Evaluation using Trajectory Match

_Judging the trajectory of an agent's execution against an expected trajectory._

In [ ]:
def test_trajectory_match(
    query: str,
    reference_trajectory: list,
    trajectory_match_mode=Literal["strict", "unordered", "subset", "superset"]):
    """
    Helper function to deterministically test trajectory match against reference trajectory.

    Parameters:
    - query (str): The input query to the agent.
    - reference_trajectory (list): The expected trajectory of messages.
    - trajectory_match_mode (str): The mode of trajectory matching. Options are "strict", "unordered", "subset", "superset".

    Returns:
    - dict: A dictionary containing the evaluation results.
    """

    # Get the model execution trajectory by calling `invoke` method of the `agent` instance
    # passing `{"messages": [HumanMessage(content=query)]}` against first positioned parameter and
    # store the output into a variable `result`.
    
    # Code here
    # ...
    
    # Create an evaluator instance by calling function `create_trajectory_match_evaluator` passing
    # variable `trajectory_match_mode` against parameter `trajectory_match_mode` and
    # store reference of the created instance into a variable `evaluator`.

    # Code here
    # ...

    return evaluator(outputs=result["messages"], reference_outputs=reference_trajectory)

**Strict mode match**

_Matching exact message structure and tool calls in the same order (message content can differ)._

In [ ]:
query = "What's the weather in San Francisco?"      # Sets end-user query

# Sets reference trajectory the agent's trajectory would be compared with
reference_trajectory = [
        HumanMessage(content="What's the weather in San Francisco?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}}
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        AIMessage(content="The weather in San Francisco is 75 degrees and sunny."),
    ]

# To match agent's execution trajectory against a reference trajectory on a "strict" mode, 
# call helper function `test_trajectory_match` passing
# variable `query` against the first positioned parameter,
# variance `reference_trajectory` against the second positioned parameter,
# value "strict" against the third positioned parameter, and
# store the output in a variable `eval_result`.

# Code here
# ...

print(eval_result)      # Prints the evaluation

Refer to the value associated with key 'score' where it indicates 100% or 0% match against value 'True' or 'False', respectively, as part of the agent trajectory evaluation.

In [ ]:
# To analyze the above evaluation score, check the trajectory manually by calling `invoke` method of the `agent` instance
# passing `{"messages": [HumanMessage(content=query)]}` against first positioned and observe the output
    
# Code here
# ...

**Unordered mode match**

_Same as "strict" mode match, but allows tool calls in any order._

In [ ]:
query = "What's happening in San Francisco today? What's the weather there?"    # Sets end-user query

# Sets reference trajectory the agent's trajectory would be compared with
reference_trajectory = [
        HumanMessage(content="What's happening in San Francisco today? What's the weather there?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}},
            {"id": "call_2", "name": "get_events", "args": {"city": "San Francisco"}},
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        ToolMessage(content="Concert at the park in San Francisco tonight.", tool_call_id="call_2"),
        AIMessage(content="Today in San Francisco: 75 degrees and sunny with a concert at the park tonight."),
    ]

# Similarly, to match agent's execution trajectory against a reference trajectory on a "unordered" mode, 
# call once again the helper function `test_trajectory_match` passing
# variable `query` against the first positioned parameter,
# variance `reference_trajectory` against the second positioned parameter,
# value "unordered" against the third positioned parameter, and
# store the output in a variable `eval_result`.

# Code here
# ...


print(eval_result)      # Prints the evaluation

In [ ]:
# To analyze the above evaluation score, check the trajectory manually by calling `invoke` method of the `agent` instance
# passing `{"messages": [HumanMessage(content=query)]}` against first positioned and observe the output
    
# Code here
# ...

## Evaluating using LLM-as-Judge
_Using LLM as a judge to evaluate the trajectory (where reference trajectory is optional unlike the trajectory match evaluation)._


In [ ]:
def test_trajectory_quality(
    query: str,
    reference_trajectory: list = None):
    """
    Helper function to assess overall quality and reasoning of the trajectory without strict expectations using an LLM as a judge.

    Parameters:
    - query (str): The input query to the agent.
    - reference_trajectory (list): The expected trajectory of messages. If provided, the judge will compare the agent's trajectory against this reference.

    Returns:
    - dict: A dictionary containing the evaluation results.
    """

    # Get the model execution trajectory by calling `invoke` method of the `agent` instance
    # passing `{"messages": [HumanMessage(content=query)]}` against first positioned parameter and
    # store the output into a variable `result`.
    
    # Code here
    # ...
    
    # Create an evaluator instance by calling function `create_trajectory_llm_as_judge` passing
    # model variable `model_as_judge` against parameter `judge` and
    # constant `TRAJECTORY_ACCURACY_PROMPT` if `reference_trajectory` is `None` otherwise
    # constant `TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE` against parameter `prompt`, and 
    # store reference of the created instance into a variable `evaluator`.

    # Code here
    # ...

    return evaluator(outputs=result["messages"])

In [ ]:
query = "What's the weather in Seattle?"

eval_result = test_trajectory_quality(query)

print(eval_result)

Refer to the values associated with both the keys 'score' and 'comment'. The 'comment' contains the explanation behind the score.